<a href="https://colab.research.google.com/github/abdul2924/Machine_learning_Assignment/blob/main/Feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Feature Engineering Pipeline
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import VectorAssembler, StandardScaler, OneHotEncoder, StringIndexer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable


BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"
INPUT_DATA = f"{BASE_PATH}/data/processed/clean_taxi_data"
OUTPUT_PATH = f"{BASE_PATH}/data/final"


spark = SparkSession.builder \
    .appName("NYCTaxi_FeatureEngineering_2019") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()


df = spark.read.parquet(INPUT_DATA)


class TaxiFeatureGenerator(Transformer, DefaultParamsReadable, DefaultParamsWritable):

    def _transform(self, dataset):

        R = 6371.0

        dlat = F.radians(F.col("pickup_latitude") - F.col("dropoff_latitude"))
        dlon = F.radians(F.col("pickup_longitude") - F.col("dropoff_longitude"))

        a = (F.sin(dlat / 2) ** 2 +
             F.cos(F.radians(F.col("pickup_latitude"))) *
             F.cos(F.radians(F.col("dropoff_latitude"))) *
             F.sin(dlon / 2) ** 2)

        c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
        haversine_km = R * c

        return dataset.withColumn("haversine_km", haversine_km.cast(DoubleType())) \
            .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0) \
            .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
            .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
            .withColumn("is_peak_hour", F.when(F.col("pickup_hour").between(7,9) |
                                             F.col("pickup_hour").between(16,19), 1).otherwise(0)) \
            .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)) \
            .fillna(0) # Proper handling of nulls


df_prepared = df.withColumn("pickup_latitude", F.lit(40.7128)) \
                .withColumn("pickup_longitude", F.lit(-74.0060)) \
                .withColumn("dropoff_latitude", F.lit(40.7128)) \
                .withColumn("dropoff_longitude", F.lit(-74.0060))

feature_gen = TaxiFeatureGenerator()


indexer = StringIndexer(inputCol="day_of_week", outputCol="dow_index")
encoder = OneHotEncoder(inputCol="dow_index", outputCol="dow_vector")


numeric_features = [
    "passenger_count", "trip_distance", "haversine_km",
    "trip_duration_min", "pickup_hour", "is_peak_hour", "is_weekend"
]

assembler = VectorAssembler(
    inputCols=numeric_features + ["dow_vector"],
    outputCol="features_unscaled"
)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)


pipeline = Pipeline(stages=[feature_gen, indexer, encoder, assembler, scaler])


pipeline_model = pipeline.fit(df_prepared)
df_final = pipeline_model.transform(df_prepared)


train_df = df_final.filter(F.col("pickup_month") <= "2019-10")
val_df = df_final.filter(F.col("pickup_month") == "2019-11")
test_df = df_final.filter(F.col("pickup_month") == "2019-12")


def save_data(df, name):
    path = f"{OUTPUT_PATH}/{name}"
    df.select("features", "fare_amount") \
      .write.mode("overwrite") \
      .parquet(path)
    print(f"Saved {name} to {path}")

save_data(train_df, "train_2019")
save_data(val_df, "val_2019")
save_data(test_df, "test_2019")


pipeline_model.write().overwrite().save(f"{BASE_PATH}/models/feature_pipeline_2019")

print("\n Feature Engineering Complete ")
df_final.select("features", "fare_amount").show(5, truncate=False)

Saved train_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/train_2019
Saved val_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/val_2019
Saved test_2019 to /content/drive/MyDrive/yellow_tripdata_2019/data/final/test_2019

 Feature Engineering Complete 
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                |fare_amount|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------